# Case Study 7.2 - Feature Engineering

In this case study we explore using language models for fetaure engineering experiments

The original python scripts used in developing this case study are provided in the following files:
* [CaseStudy_7.2_01-01.py](CaseStudy_7.2_01-01.py)
* [CaseStudy_7.2_01-02.py](CaseStudy_7.2_01-02.py)
* [CaseStudy_7.2_01-03.py](CaseStudy_7.2_01-01.py)
* [CaseStudy_7.2_01-04.py](CaseStudy_7.2_01-04.py)
  

In [ ]:
import json
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
class Essay_Features(BaseModel):
    embedded_clause: int = Field(
        description="Count of the number of embedded clauses."
    )
    passive_voice: int = Field(
        description="Count of the instances of passive voice."
    )
    long_sentences: int = Field(
        description="Count of sentences longer than 20 words."
    )
    obscure_vocab: int = Field(
        description="Count of obscure vocabulary used."
    )
    reasoning_lines: int = Field(
        description="Count of the number of lines of reasoning used."
    )
    unusual_grammar: int = Field(
        description="Count of unusual grammatical constructs."
    )
    has_adjectives: int = Field(
        description="Indicator variable for presence of adjectives"
    )
    has_adverbs: int = Field(
        description="Indicator variable for presence of adverbs."
    )
    has_synonyms: int = Field(
        description="Indicator variable for presence of synonyms."
    )
    repeated_words: int  = Field(
        description="Count of the number of times key semantic words are\
                     repeated (not including stop words)"
    )
    detailed_vocab: int  = Field(
        description="Count of the number of detailed words used, they\
                     should be well known but very specific words e.g.\
                     ecstatic, depressed, or resentful to describe a mood."
    )
    rare_vocab: int  = Field(
        description="Indicator variable to show that the text contains\
                     rare words, these are detailed words that are\
                     uncommon in everday usage. e.g.\
                     elated, solemn or vindictive to describe a mood."
    )
    literary_vocab: int = Field(
        description="Indicator variable to show taht the text contains\
                     some literary words, these should be words that are\
                     typically only used in literary\
                     writing e.g. chagrin, enuui or vicissitudes."
    )
    turn_of_phrase: int = Field(
        description="Indicator variable to show in the text contains\
                     unique (non-cliche) turns of phrase. These should\
                     be a short expressive phrases used in the text."
    )
    disconnections: int = Field(
        description="Count of the number of times a sentence does not flow\
                    logically from the preceding text block."
    )
    contradictions: int = Field(
        description="Count of the number of contradictions in the text."
    )
    tense_discontinuity: int = Field(
        description="Count of number of times that the text changes tense,\
        via verb conjugation."
    )


parser = JsonOutputParser(pydantic_object=Essay_Features)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
   template = """
     You are an English language expert, your job is to process a student
     essay and calculate multiple numerical features that evaluate the
     quality of their writing.
     These features involve looking at the grammatical elements of the
     text, the choice of words used and the logical structure of the
     argument or story that is presented.

     Here is data for you to process:
     ---
     Essay: {essay}
     ---
     {format_instructions}
     """,
   input_variables=["essay"],
   partial_variables={
      "format_instructions":parser.get_format_instructions()
   }
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
model_name = "gemini-2.5-pro"
model = ChatGoogleGenerativeAI(model=model_name)
chain = prompt | model | parser

test_essays = [
"""
Arriving in a new town, which can be both exciting and daunting, brings with it a sense of adventure. As I stepped off the bus, which had been delayed for hours, I immediately noticed the bustling streets filled with unfamiliar faces. Finding my way to the hotel, where I would be staying for the next few days, proved to be more challenging than I could possibly have anticipated. Despite feeling a bit bamboozled, I couldn't help but marvel at the phenomenal architecture that adorned the buildings lining the cobblestone streets. Eventually, I stumbled upon a quaint Parisian cafe, where I decided to take a moment to sip tea, soak in the atmosphere and plan my next steps.
""",
"""
Arriving in a new town brings with it a sense of adventure. I stepped off the bus and saw streets filled with people. Finding my way to the hotel was hard. I felt a bit lost, but still managed to admire the buildings. I found a small cafe and planned my next steps.
"""
]

for essay in test_essays:
    response = chain.invoke({"essay": essay})
    print(response)
    print("\n")

## 02 - Custom Chain Invocation

In the next section we create a simple custom function that can be included in the sequence of LangChain operations. This allows us to include some arbitrary custom processing with the language model invocations.

In [ ]:
from typing import Callable, Dict
from langchain_core.runnables import RunnableSequence

def custom_invoke (
  chain:RunnableSequence,
  var: str, text:str,
  funcs: list[(str,Callable)]
 ):
    response = chain.invoke({var:text})
    for f in funcs:
        response[f[0]] = f[1](text, response)
    return response


In [ ]:
def sentence_count(x:str, r:Dict):
    return len(x.split("."))


def word_count(x:str, r:Dict):
    return len(x.split(" "))

In [ ]:
def readbility_score(x:str, r:Dict):
    probs = 2*(r['embedded_clause']/r['sentence_count'])+\
            (r['passive_voice']/r['sentence_count'])+\
            2*(r['long_sentences']/r['sentence_count'])+\
            (r['obscure_vocab']/(r['word_count']/2))+\
            2*(r['reasoning_lines']/r['sentence_count'])+\
            (r['unusual_grammar']/r['sentence_count'])
    if probs>10:
        return 0
    else:
        return 10 - probs

In [ ]:
def vocab_score(x:str, r:Dict):
    base_vocab = r['has_adjectives']+r['has_adverbs']+r['has_synonyms']
    detail_vocab = 4*(r['detailed_vocab']/(r['word_count']/2))
    bonus_vocab= r['rare_vocab']+r['literary_vocab']+r['turn_of_phrase']
    repeats = int(r['repeated_words']/(r['word_count']/2))
    score = base_vocab + detail_vocab + bonus_vocab - repeats
    return score

In [ ]:
procs = [
   ("word_count", word_count),
   ("sentence_count", sentence_count),
   ("readability", readbility_score),
   ("vocabulary", vocab_score)
]

for essay in test_essays:
    response = custom_invoke(chain, "essay", essay, procs)
    print(response)
    print("\n")

## 03 - Text Extraction

We change our approach to prompt the model to return the identified text segments for each of the features.

This involves redefining the PyDantic class from the top of the notebook, and including new fields that describe the extracted text for each of the numeric fields that previously just counted the number of instances of a text feature.

In [ ]:
import json
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class Essay_Features(BaseModel):
    embedded_clause: int = Field(
        description="Count of the number of embedded clauses."
    )
    passive_voice_use: list[str] = Field(
        description="Extracted instances of verbs in passive voice from the text.\
                     These should be any instances of a verb in which a verb phrase is passive."
    )
    passive_voice: int = Field(
        description="Count of the instances of passive voice."
    )
    long_sentences: int = Field(
        description="Count of sentences longer than 20 words."
    )
    obscure_vocab: int = Field(
        description="Count of obscure vocabulary used."
    )
    obscure_vocab_use: list[str] = Field(
        description="Extracted words from the text that are considered obscure vocabulary."
    )
    reasoning_lines: int = Field(
        description="Count of the number of lines of reasoning used."
    )
    unusual_grammar: int = Field(
        description="Count of unusual grammatical constructs."
    )
    has_adjectives: int = Field(
        description="Indicator variable for presence of adjectives"
    )
    has_adverbs: int = Field(
        description="Indicator variable for presence of adverbs."
    )
    has_synonyms: int = Field(
        description="Indicator variable for presence of synonyms."
    )
    repeated_words: int  = Field(
        description="Count of the number of times key semantic words are\
                     repeated (not including stop words)"
    )
    repeated_words_use: list[str] = Field(
        description="Extracted instances of repeated words in the text.\
                     These should be meaningful and significant words not including\
                     stop words such as prepositions and pronouns."
    )
    detailed_vocab: int  = Field(
        description="Count of the number of detailed words used, they\
                     should be well known but very specific words e.g.\
                     ecstatic, depressed, or resentful to describe a mood."
    )
    detailed_vocab_used: list[str]  = Field(
        description="Extracted instances of detailed vocabulary words used in the text.\
                     These should be widely known but very specific words e.g.\
                     ecstatic, depressed, or resentful to describe a mood."
    )
    rare_vocab: int  = Field(
        description="Indicator variable to show that the text contains rare words, these are\
                     detailed words that are uncommon in everday usage e.g.\
                     elated, solemn or vindictive to describe a mood."
    )
    rare_vocab_used: list[str]  = Field(
        description="Extracted instances of rare vocabulary words used in the text.\
                     These words should be very specific in meaning and uncommon in everday usage e.g.\
                     elated, solemn or vindictive to describe a mood."
    )
    literary_vocab: int = Field(
        description="Indicator variable to show taht the text contains some literary words, these\
                     should be words that typically only used in literary\
                     writing e.g. chagrin, enuui or vicissitudes."
    )
    literary_vocab_used: list[str] = Field(
        description="Extracted instances of literary words used in the text.\
                     These should be words that typically only used in literary\
                     writing e.g. chagrin, enuui or vicissitudes."
    )
    turn_of_phrase: int = Field(
        description="Indicator variable to show in the text contains unique (non-cliche) turns of\
                     phrase. These should be a short expressive phrases used in the text."
    )
    turn_of_phrase_used: list[str] = Field(
        description="Extracted instances unique (non-cliche) turns of\
                     phrase used in the text. Can be quotations as long as they are not common."
    )
    disconnections: int = Field(
        description="Count of the number of times a sentence does not flow\
                    logically from the preceding text block."
    )
    contradictions: int = Field(
        description="Count of the number of contradictions in the text."
    )
    tense_discontinuity: int = Field(
        description="Count of number of times that the text changes tense,\
        via verb conjugation."
    )


parser = JsonOutputParser(pydantic_object=Essay_Features)

## 04 - Generated Feature Functions

We prompted a proprietary model to generate a series of functions for creatingf standard text evaluation functions. We deliberately made these compatible with our `custom_invoke` function so we could include them in the completed pipeline. The generated functions are saved in a file called [claude_feature_funcs.py](claude_feature_funcs.py) and then included as follows:

In [ ]:
import claude_feature_funcs as claude

procs = [
   ("word_count", word_count),
   ("sentence_count", sentence_count),
   ("readability", readbility_score),
   ("vocabulary", vocab_score),
   ("flesch_reading_ease", claude.flesch_reading_ease),
   ("flesch_kincaid_grade_level", claude.flesch_kincaid_grade_level),
   ("automated_readability_index", claude.automated_readability_index),
   ("gunning_fog_index", claude.gunning_fog_index),
   ("smog_readability", claude.smog_readability),
   ("type_token_ratio", claude.type_token_ratio),
   ("moving_average_ttr", claude.moving_average_ttr),
   ("yule_k_characteristic", claude.yule_k_characteristic),
   ("average_word_length", claude.average_word_length),
   ("syllable_complexity", claude.syllable_complexity),
   ("logical_connector_density", claude.logical_connector_density),
   ("argument_structure_score", claude.argument_structure_score),
   ("coherence_score", claude.coherence_score),
   ("contradiction_detection_score", claude.contradiction_detection_score)
]

## 05 - Final Script

The ideas in this case study were compiled into a [single command line application](essay_features.py) to process a dataset of essays and produce all features for each record in the data. That script can be invoked as follows:

```
uv run python essay_features.py data/Essays.csv data/Essays_scored.csv
```

You can then evaluate the features using a script that looks at correlation between our generated features and the established quantitative methos of evaluating writing. Ideally you want to see some correlation (indicting coherenece in what is being measured), but also sufficient independence to indicate that you features could have novel information value.

```
uv run python analyse_essays.py data/Essays_scored.csv
```
